# DIF Simulation 2: Improved Bayesian IRT Model
## 능력 차이(Impact)와 문항 편향(DIF)의 통계적 분리

이 노트북은 기존 분석의 신뢰성 결함을 해결하기 위해, 집단 간 능력 평균 차이를 직접 추정하는 개선된 Stan 모델을 사용합니다.

### 핵심 개선 사항:
1. **Impact 모델링:** 초점 집단의 평균 능력($\mu_{focal}$)을 자유 파라미터로 설정하여, 집단의 실력 차이가 문항 편향($\delta$)으로 오염되는 것을 방지합니다.
2. **신뢰성 검증:** 이전 모델에서 발생했던 다수의 False Positive(위양성)가 어떻게 줄어드는지 확인합니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.special import expit
from cmdstanpy import CmdStanModel
import os

sns.set_theme(style="whitegrid")
RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

# 데이터 생성 (기존과 동일한 조건)
N_REF, N_FOCAL = 150, 120
N_ITEMS = 20

true_beta = np.linspace(-1.8, 1.8, N_ITEMS)
true_delta = np.zeros(N_ITEMS)
DIF_ITEMS = {3: 0.9, 9: 1.1, 10: 0.7, 16: -0.8, 19: 1.2} # 5개 문항에만 DIF 주입
for idx, val in DIF_ITEMS.items():
    true_delta[idx] = val

# 집단 간 실제 실력 차이 주입 (참조=0.3, 초점=-0.4)
theta_ref = rng.normal(0.3, 1.0, N_REF)
theta_focal = rng.normal(-0.4, 1.0, N_FOCAL)

def simulate(theta, beta, delta, group_id):
    probs = expit(theta[:, None] - (beta[None, :] + delta[None, :] * group_id))
    resp = rng.binomial(1, probs)
    df = pd.DataFrame(resp, columns=[f"item{j+1:02d}" for j in range(N_ITEMS)])
    df["group"] = group_id
    return df

df_all = pd.concat([simulate(theta_ref, true_beta, true_delta, 0), 
                    simulate(theta_focal, true_beta, true_delta, 1)], ignore_index=True)

print(f"데이터 생성 완료: 총 {len(df_all)}명, DIF 문항 {len(DIF_ITEMS)}개")

## 개선된 Stan 모델 컴파일 및 피팅
`mu_focal` 파라미터가 추가된 `dif_rasch_improved.stan` 모델을 사용합니다.

In [ ]:
stan_file = "stan_models/dif_rasch_improved.stan"
model = CmdStanModel(stan_file=stan_file)

stan_data = {
    "N": len(df_all),
    "J": N_ITEMS,
    "Y": df_all.filter(like="item").values.tolist(),
    "group": df_all["group"].tolist()
}

fit = model.sample(data=stan_data, seed=RNG_SEED, show_progress=True)
summary = fit.summary()

## 결과 분석 및 신뢰성 검증

In [ ]:
delta_samples = fit.stan_variable("delta")
mu_focal_samples = fit.stan_variable("mu_focal")

print(f"추정된 초점 집단 평균 능력 (True=-0.7 relative to Ref=0.3): {mu_focal_samples.mean():.3f}")

results = []
for j in range(N_ITEMS):
    d_samples = delta_samples[:, j]
    prob_dif = np.mean(np.abs(d_samples) >= 0.2)
    results.append({
        "item": f"item{j+1:02d}",
        "true_delta": true_delta[j],
        "est_delta": d_samples.mean(),
        "prob_dif": prob_dif,
        "detected": prob_dif >= 0.8
    })

res_df = pd.DataFrame(results)
res_df["actual_dif"] = res_df["true_delta"] != 0

# Confusion Matrix 계산
tp = ((res_df.detected) & (res_df.actual_dif)).sum()
fp = ((res_df.detected) & (~res_df.actual_dif)).sum()
fn = ((~res_df.detected) & (res_df.actual_dif)).sum()
tn = ((~res_df.detected) & (~res_df.actual_dif)).sum()

print(f"\n[DIF 탐지 결과 Comparison]")
print(f"True Positives: {tp}")
print(f"False Positives: {fp} (이전 모델 대비 감소 확인)")
print(f"Precision: {tp/(tp+fp) if tp+fp>0 else 0:.2f}")
print(f"Recall: {tp/(tp+fn) if tp+fn>0 else 0:.2f}")

## 시각화: 개선된 DIF 파라미터 추정치

In [ ]:
plt.figure(figsize=(10, 6))
plt.errorbar(res_df.index, res_df.est_delta, yerr=res_df.est_delta*0.1, fmt='o', label='Estimated Delta')
plt.scatter(res_df.index, res_df.true_delta, color='red', marker='x', label='True Delta', zorder=5)
plt.axhline(0.2, color='gray', linestyle='--')
plt.axhline(-0.2, color='gray', linestyle='--')
plt.title("Improved Model: DIF Parameter Recovery")
plt.xlabel("Item Index")
plt.ylabel("Delta (DIF Effect)")
plt.legend()
plt.show()